# Exploratory Data Analysis (EDA) & Data Visualization
## Project: Fresh and Rotten Fruit/Vegetable Classification

This notebook provides a comprehensive exploratory data analysis (EDA) of the fruit and vegetable dataset, including statistical summaries, class distributions, subcategory analysis, and image properties (resolution, aspect ratio).

### 1. Import Libraries

In [ ]:
import os
import random
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

# Set style for visualization
sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.titlesize': 16})

### 2. Configure Dataset Directory
Please update `DATA_DIR` below to point to your local dataset path if it is located elsewhere.

In [ ]:
DATA_DIR = Path('dataset')
print(f'Checking dataset path: {DATA_DIR.resolve()}')
print(f'Exists: {DATA_DIR.exists()}')

### 3. Scan Dataset and Build DataFrame
We will scan the `Train` and `Test` directories, extract the class label (Fresh/Rotten), subcategory (apples, bananas, oranges, etc.), and collect image details (width, height, aspect ratio).

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

data_records = []

if not DATA_DIR.exists():
    print('WARNING: Dataset folder not found. Please verify the path!')
else:
    for split in ['Train', 'Test']:
        split_dir = DATA_DIR / split
        if not split_dir.exists():
            continue
        
        for folder in split_dir.iterdir():
            if not folder.is_dir():
                continue
            
            folder_name = folder.name.lower()
            
            # Determine label
            if 'fresh' in folder_name:
                label = 'Fresh'
            elif 'rotten' in folder_name or 'stale' in folder_name:
                label = 'Rotten'
            else:
                label = 'Unknown'
            
            # Determine subcategory (e.g. apples, banana, cucumber, etc.)
            # Stripping 'fresh' and 'rotten' prefixes to unify the fruit/veg category name
            subcat = folder_name.replace('fresh', '').replace('rotten', '').replace('stale', '')
            if subcat == 'patato': # fix typo in directory names if any
                subcat = 'potato'
            if subcat == 'tamto':
                subcat = 'tomato'
            
            for file_path in folder.glob('**/*'):
                if file_path.suffix.lower() in IMG_EXTS:
                    try:
                        # Open image header to get size without reading full pixel data
                        with Image.open(file_path) as img:
                            width, height = img.size
                        
                        data_records.append({
                            'path': str(file_path),
                            'split': split,
                            'folder': folder.name,
                            'label': label,
                            'subcategory': subcat,
                            'width': width,
                            'height': height,
                            'aspect_ratio': width / height,
                            'pixels': width * height
                        })
                    except Exception as e:
                        print(f'Error reading {file_path}: {e}')

df = pd.DataFrame(data_records)
print(f'Scan complete! Loaded {len(df)} images.')
if not df.empty:
    display(df.head())

### 4. Basic Dataset Statistics

In [ ]:
if df.empty:
    print('No data loaded. Cannot generate statistics.')
else:
    print('--- Dataset Overview ---')
    print(f'Total Images: {len(df)}')
    print(f'Splits: {df["split"].value_counts().to_dict()}')
    print(f'Labels: {df["label"].value_counts().to_dict()}')
    print(f'Unique Subcategories ({len(df["subcategory"].unique())}): {sorted(df["subcategory"].unique())}')
    print("\n--- Image Dimensions Summary ---")
    display(df[['width', 'height', 'aspect_ratio']].describe())

### 5. Data Visualizations
We will visualize:
1. The balance between Fresh and Rotten samples across the Train and Test sets.
2. Subcategory frequency counts.
3. Joint distribution of image widths and heights.
4. Distribution of aspect ratios.

In [ ]:
if df.empty:
    print('No data loaded. Cannot plot.')
else:
    # Set up matplotlib figure
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    
    # Plot 1: Split vs Label Count
    sns.countplot(data=df, x='split', hue='label', palette='Set2', ax=axes[0,0])
    axes[0,0].set_title('Class Distribution: Train vs Test Set')
    axes[0,0].set_xlabel('Dataset Split')
    axes[0,0].set_ylabel('Number of Images')
    
    # Plot 2: Subcategory Counts
    subcat_counts = df['subcategory'].value_counts()
    sns.barplot(x=subcat_counts.values, y=subcat_counts.index, palette='viridis', ax=axes[0,1])
    axes[0,1].set_title('Distribution by Category (Fruit/Vegetable)')
    axes[0,1].set_xlabel('Count')
    axes[0,1].set_ylabel('Category')
    
    # Plot 3: Scatter plot of image dimensions
    # Sample a subset to avoid overplotting if dataset is too large
    sample_df = df.sample(min(2000, len(df)), random_state=42)
    sns.scatterplot(data=sample_df, x='width', y='height', hue='label', alpha=0.6, palette='Set1', ax=axes[1,0])
    axes[1,0].set_title('Image Resolution Distribution (Sampled)')
    axes[1,0].set_xlabel('Width (px)')
    axes[1,0].set_ylabel('Height (px)')
    
    # Plot 4: Aspect Ratio Distribution
    sns.histplot(data=df, x='aspect_ratio', hue='label', kde=True, bins=30, palette='Set2', multiple='stack', ax=axes[1,1])
    axes[1,1].set_title('Aspect Ratio Distribution')
    axes[1,1].set_xlabel('Aspect Ratio (Width / Height)')
    axes[1,1].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()

### 6. Subcategory Analysis: Fresh vs. Rotten
Let's see how each specific item category is split between Fresh and Rotten.

In [ ]:
if df.empty:
    print('No data loaded.')
else:
    plt.figure(figsize=(12, 8))
    # Order categories alphabetically or by frequency
    order = sorted(df['subcategory'].unique())
    sns.countplot(data=df, y='subcategory', hue='label', order=order, palette='coolwarm')
    plt.title('Fresh vs. Rotten Counts by Category')
    plt.xlabel('Count')
    plt.ylabel('Category')
    plt.legend(title='Condition')
    plt.tight_layout()
    plt.show()

### 7. Visualize Sample Images from the Dataset

In [ ]:
if df.empty:
    print('No images to display.')
else:
    # Select 8 random images
    sample_rows = df.sample(8, random_state=random.randint(1, 1000))
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 9))
    axes = axes.flatten()
    
    for i, (_, row) in enumerate(sample_rows.iterrows()):
        img_path = row['path']
        label = row['label']
        subcat = row['subcategory']
        width = row['width']
        height = row['height']
        split = row['split']
        
        img = Image.open(img_path)
        axes[i].imshow(img)
        axes[i].axis('off')
        
        # Color code for label
        color = 'green' if label == 'Fresh' else 'red'
        
        axes[i].set_title(
            f'{subcat.capitalize()} ({split})\nLabel: {label}\n{width}x{height} px',
            color=color,
            fontsize=11,
            fontweight='bold'
        )
        
    plt.tight_layout()
    plt.show()